<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/2026-2027/7_3_Nonlinear_Dimensionality_Reduction_t%E2%80%91SNE%2C_UMAP%2C_Isomap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part VII. Dimensionality Reduction, Unsupervised Learning, and Geometry of Data

## 1. Curse and Blessings of Dimensionality    (Geometry of the d‑dimensional sphere and cube, concentration of measure, condition number of random matrices)
## 2. Principal Component Analysis (PCA) and SVD
## 3. Nonlinear Dimensionality Reduction: t‑SNE, UMAP, Isomap
## 4. Clustering: K‑means, Hierarchical, DBSCAN


## Когда линейное снижение размерности подводит: нелинейные многообразия

Метод главных компонент (PCA) — мощный инструмент, но его фундаментальное ограничение заложено в названии: он ищет **линейные** комбинации исходных признаков. Когда данные разбросаны вдоль прямых или плоскостей, PCA работает безупречно. Однако в реальном мире данные часто лежат на изогнутых, скрученных поверхностях — **многообразиях**, которые лишь локально напоминают евклидово пространство. Представьте себе смятый лист бумаги: в трёхмерном пространстве он занимает сложную форму, но по сути остаётся двумерным объектом. Чтобы понять, что на нём написано, лист нужно аккуратно распрямить, сохранив взаимное расположение точек. Именно эту задачу — «распрямление» без разрывов — решают нелинейные методы снижения размерности.

### 1. Ограничения PCA на изогнутых поверхностях

PCA ищет подпространство, максимизирующее глобальную дисперсию. Если данные образуют спираль или подкову, направления с наибольшей дисперсией могут проходить поперёк многообразия, перемешивая участки, которые на самом деле далеки друг от друга вдоль поверхности. В результате точки, удалённые в смысле геодезического расстояния (кратчайшего пути по многообразию), могут оказаться рядом в проекции, а близкие — далеко. PCA «разрезает» и «склеивает» многообразие, разрушая его истинную структуру. Это делает его непригодным для визуализации и анализа данных, где важна не глобальная линейная вариация, а локальные отношения соседства.

### 2. Классические примеры нелинейных многообразий

Чтобы наглядно увидеть проблему, рассмотрим два искусственных, но очень показательных датасета: **Swiss roll** (швейцарский рулет) и **S-образную кривую**. Оба генерируются в `sklearn.datasets`.

**Swiss roll** — это двумерный лист, свёрнутый в трёхмерную спираль. Истинная структура — плоский прямоугольник, где координаты вдоль и поперёк рулета являются естественными параметрами. Если применить PCA и спроецировать на первые две главные компоненты, мы не получим развёртки: спираль будет сплющена в бесформенное облако, где слои рулета наложатся друг на друга.

**S-образная кривая** — двумерная поверхность, изогнутая в трёхмерном пространстве в форме буквы S. PCA опять же не сможет корректно «расправить» S-форму, так как для этого требуется нелинейное преобразование.

Ниже мы сгенерируем оба набора данных и визуализируем результаты PCA.

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_swiss_roll, make_s_curve
from sklearn.decomposition import PCA
from mpl_toolkits.mplot3d import Axes3D

# Генерация Swiss roll
n_samples = 1000
X_swiss, t_swiss = make_swiss_roll(n_samples=n_samples, noise=0.0)

# Генерация S-curve
X_scurve, t_scurve = make_s_curve(n_samples=n_samples, noise=0.0)

# Функция для визуализации трёхмерных данных и их PCA-проекций
def plot_original_and_pca(X, labels, title_3d, title_pca, azim=-60, elev=30):
    fig = plt.figure(figsize=(12,5))
    
    # 3D-график
    ax1 = fig.add_subplot(1,2,1, projection='3d')
    ax1.scatter(X[:,0], X[:,1], X[:,2], c=labels, cmap='Spectral', s=10)
    ax1.view_init(elev=elev, azim=azim)
    ax1.set_title(title_3d, fontsize=12)
    
    # PCA-проекция на 2 компоненты
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X)
    ax2 = fig.add_subplot(1,2,2)
    ax2.scatter(X_pca[:,0], X_pca[:,1], c=labels, cmap='Spectral', s=10)
    ax2.set_title(title_pca, fontsize=12)
    ax2.set_xlabel('PC1')
    ax2.set_ylabel('PC2')
    plt.tight_layout()
    plt.show()

# Swiss roll: цвет соответствует координате вдоль рулета
plot_original_and_pca(X_swiss, t_swiss,
                      'Swiss roll в 3D', 'PCA проекция Swiss roll (2D)',
                      azim=-60, elev=10)

# S-curve: цвет соответствует координате вдоль кривой
plot_original_and_pca(X_scurve, t_scurve,
                      'S-образная кривая в 3D', 'PCA проекция S-curve (2D)',
                      azim=-120, elev=15)
```

На рисунках отчётливо видно, что PCA не способен «развернуть» рулет или распрямить S-кривую. Цвет, отражающий истинную параметризацию многообразия, оказывается перемешанным в проекции PCA. Это означает, что евклидовы расстояния в трёхмерном пространстве не отражают истинной близости точек вдоль поверхности.

### 3. Интуиция: что нужно сохранять при снижении размерности

Задача нелинейного снижения размерности — построить такое отображение из исходного пространства $\mathbb{R}^d$ в пространство низкой размерности $\mathbb{R}^k$ (обычно $k=2$ или $3$ для визуализации), которое сохраняло бы **существенную структуру** данных. Но какую именно структуру?

В отличие от PCA, который фокусируется на глобальной дисперсии, нелинейные методы стараются сохранить **локальные отношения соседства**: точки, которые были близки на многообразии (в смысле геодезического расстояния), должны остаться близкими в проекции. При этом большие евклидовы расстояния, возникающие из-за кривизны поверхности, игнорируются или сильно штрафуются. Такой подход подобен аккуратному разглаживанию смятого листа: мы хотим, чтобы соседние буквы остались рядом, а разрывы не возникли.

Ключевое понятие — **геодезическое расстояние**. В отличие от евклидова расстояния, которое измеряет длину отрезка напрямую сквозь пространство, геодезическое расстояние идёт вдоль поверхности. Например, на Swiss roll расстояние между двумя точками, лежащими в соседних слоях спирали, по прямой мало, но геодезически — велико, так как нужно пройти по спирали. Именно геодезические расстояния должны быть по возможности сохранены.

### 4. Формальная постановка задачи

Дана матрица данных $\mathbf{X} \in \mathbb{R}^{n \times d}$, где $n$ — число точек, $d$ — размерность исходного пространства. Мы ищем матрицу $\mathbf{Y} \in \mathbb{R}^{n \times k}$, $k \ll d$, такую, что в ней сохранены структурные свойства $\mathbf{X}$. Для нелинейных методов $\mathbf{Y}$ не является линейной проекцией: нельзя записать $\mathbf{Y} = \mathbf{X}\mathbf{W}$ с некоторой матрицей $\mathbf{W}$. Вместо этого отображение строится на основе анализа попарных расстояний или вероятностных моделей соседства.

Все рассматриваемые нами методы — **Isomap**, **t‑SNE** и **UMAP** — решают эту задачу, но каждый по-своему. Общим для них является то, что они работают не с абсолютными координатами, а с информацией о близости точек.

### 5. Краткий обзор методов (анонс)

Прежде чем углубляться в детали каждого алгоритма, наметим их ключевые идеи.

- **Isomap** (Isometric Feature Mapping) расширяет классическое многомерное шкалирование (MDS), заменяя евклидовы расстояния на геодезические. Геодезические расстояния аппроксимируются по графу ближайших соседей (алгоритм Флойда–Уоршелла или Дейкстры). Затем MDS находит низкоразмерное представление, которое лучше всего сохраняет эти попарные геодезические расстояния. Isomap особенно хорошо работает, когда многообразие изометрично евклидову пространству (разворачивается без искажений), как в случае Swiss roll.

- **t‑SNE** (t‑distributed Stochastic Neighbor Embedding) — вероятностный метод, который преобразует евклидовы расстояния в исходном пространстве в условные вероятности, отражающие «сходство» точек, а затем пытается найти низкоразмерное представление, в котором распределение сходства максимально близко к исходному в смысле дивергенции Кульбака–Лейблера. Для избежания скученности в низком измерении используется t‑распределение Стьюдента. t‑SNE превосходно визуализирует кластеры, но не сохраняет глобальную геометрию.

- **UMAP** (Uniform Manifold Approximation and Projection) строит нечёткое топологическое представление данных с помощью теории симплициальных множеств и затем оптимизирует низкоразмерное вложение, минимизируя кросс-энтропию между топологическими представлениями высокого и низкого измерений. UMAP часто работает быстрее t‑SNE и лучше сохраняет как локальную, так и глобальную структуру.

В следующих разделах мы подробно рассмотрим каждый из этих алгоритмов, их математические основы и практическую реализацию в Python.

---

### Резюме

Линейные методы, такие как PCA, отлично справляются с данными, имеющими глобальные направления максимальной дисперсии, но терпят неудачу, когда данные лежат на изогнутых нелинейных многообразиях. Для визуализации и анализа таких данных необходимы нелинейные методы, которые сохраняют локальную структуру и геодезические расстояния. Isomap, t‑SNE и UMAP предлагают три разных подхода к этой задаче, и их изучение откроет путь к пониманию сложных многомерных наборов данных.

**Вопросы для самопроверки:**
1. Почему PCA перемешивает слои Swiss roll при проекции на две главные компоненты? Что именно теряется?
2. Чем геодезическое расстояние отличается от евклидова на примере S‑образной кривой? Почему для разворачивания многообразия важно использовать именно геодезические расстояния?

## Isomap: изометрическое отображение

Isomap (Isometric Feature Mapping) — один из первых и наиболее интуитивно понятных методов нелинейного снижения размерности. Предложенный Тененбаумом, де Сильвой и Лэнгфордом в 2000 году, он расширяет классическое многомерное шкалирование (MDS), заменяя евклидовы расстояния на **геодезические**. Идея проста: если данные лежат на изогнутом многообразии, то истинное расстояние между точками — это длина кратчайшего пути вдоль этого многообразия, а не расстояние по прямой сквозь пространство. Isomap приближает геодезические расстояния с помощью графа ближайших соседей и затем строит низкоразмерное вложение, максимально сохраняющее эти расстояния.

### 1. Интуиция и математическая постановка

Пусть данные $\mathbf{X} = \{\mathbf{x}_1,\dots,\mathbf{x}_n\} \subset \mathbb{R}^d$ лежат на гладком многообразии $\mathcal{M}$ размерности $k \ll d$. Мы хотим найти отображение $\mathbf{x}_i \mapsto \mathbf{y}_i \in \mathbb{R}^k$, которое сохраняет **геодезические расстояния** $d_{\mathcal{M}}(\mathbf{x}_i, \mathbf{x}_j)$ как можно точнее, то есть

$$
\|\mathbf{y}_i - \mathbf{y}_j\| \approx d_{\mathcal{M}}(\mathbf{x}_i, \mathbf{x}_j), \quad \forall i,j.
$$

Если $\mathcal{M}$ — изометрично евклидову пространству, то такое представление $\mathbf{Y}$ существует и может быть найдено методом MDS, применённым к матрице истинных геодезических расстояний. Ключевая проблема — оценить геодезические расстояния, имея лишь конечный набор точек. Isomap делает это с помощью двух приближений:
- **Локальная линейность:** в малой окрестности многообразие можно считать плоским, и евклидово расстояние между близкими точками совпадает с геодезическим.
- **Графовая аппроксимация:** глобальное геодезическое расстояние аппроксимируется длиной кратчайшего пути в графе соседей, построенном на данных.

### 2. Алгоритм Isomap по шагам

**Шаг 1. Построение графа соседей.**  
Для каждой точки $\mathbf{x}_i$ находим её $k$ ближайших соседей (k-NN) или все точки в радиусе $\varepsilon$ ($\varepsilon$-окрестность). Соединяем рёбрами $\mathbf{x}_i$ и $\mathbf{x}_j$, если $j$ — сосед $i$ или $i$ — сосед $j$. Вес ребра полагаем равным евклидову расстоянию $\|\mathbf{x}_i - \mathbf{x}_j\|$. Такой граф является симметричным и неориентированным. Если данные лежат на многообразии, то при достаточно большой плотности выборки этот граф аппроксимирует геодезическую метрику.

**Шаг 2. Вычисление матрицы геодезических расстояний $\mathbf{D}_G$.**  
Для каждой пары вершин $(i,j)$ находим длину кратчайшего пути в графе. Если граф несвязен, то пары из разных компонент получают расстояние $\infty$ (на практике их исключают или назначают большое значение). Эффективно можно использовать алгоритм Дейкстры для каждой вершины или алгоритм Флойда–Уоршелла (для плотных матриц). Результат — матрица $\mathbf{D}_G \in \mathbb{R}^{n \times n}$, где $D_G[i,j]$ — аппроксимированное геодезическое расстояние.

**Шаг 3. Двойное центрирование.**  
Классическое многомерное шкалирование (MDS) работает с матрицей скалярных произведений, а не с расстояниями. Чтобы перейти от $\mathbf{D}_G$ к матрице Грама $\mathbf{B}$, выполняют **двойное центрирование**:

$$
\mathbf{B} = -\frac{1}{2} \mathbf{J} \mathbf{D}_G^{\circ 2} \mathbf{J},
$$

где $\mathbf{D}_G^{\circ 2}$ — поэлементное возведение в квадрат, а $\mathbf{J} = \mathbf{I}_n - \frac{1}{n}\mathbf{1}\mathbf{1}^T$ — центрирующая матрица ($\mathbf{1}$ — вектор из единиц). Матрица $\mathbf{B}$ является симметричной и положительно полуопределённой, если $\mathbf{D}_G$ — матрица евклидовых расстояний. Для приближённых геодезических расстояний это свойство может нарушаться, но на практике $\mathbf{B}$ всё равно даёт разумное представление.

**Шаг 4. Спектральное разложение и построение вложения.**  
Находим собственные значения и собственные векторы матрицы $\mathbf{B}$: $\mathbf{B} = \mathbf{V} \boldsymbol{\Lambda} \mathbf{V}^T$, где $\boldsymbol{\Lambda} = \operatorname{diag}(\lambda_1,\dots,\lambda_n)$, $\lambda_1 \ge \lambda_2 \ge \dots \ge \lambda_n$. Для $k$-мерного вложения берём первые $k$ собственных значений и соответствующие им собственные векторы. Координаты точек в новом пространстве:

$$
\mathbf{Y} = \mathbf{V}_k \boldsymbol{\Lambda}_k^{1/2},
$$

где $\mathbf{V}_k \in \mathbb{R}^{n \times k}$ — первые $k$ собственных векторов, а $\boldsymbol{\Lambda}_k^{1/2} = \operatorname{diag}(\sqrt{\lambda_1},\dots,\sqrt{\lambda_k})$. Именно такое преобразование гарантирует, что евклидовы расстояния в $\mathbf{Y}$ будут максимально близки к исходным расстояниям $\mathbf{D}_G$ в смысле наименьших квадратов (это основной результат MDS).

### 3. Реализация на Python

Реализуем полный алгоритм Isomap, используя `scipy.spatial.cKDTree` для быстрого поиска соседей и `scipy.sparse.csgraph.shortest_path` для поиска кратчайших путей. Затем применим его к Swiss roll и сравним с PCA.

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_swiss_roll, load_digits
from sklearn.decomposition import PCA
from scipy.spatial import cKDTree
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import shortest_path, connected_components

def isomap(X, n_neighbors=10, n_components=2):
    """
    Isomap: изометрическое отображение.
    
    Параметры:
        X : ndarray (n, d) — исходные данные.
        n_neighbors : int — число соседей для построения графа.
        n_components : int — целевая размерность.
    
    Возвращает:
        Y : ndarray (n, n_components) — координаты вложения.
    """
    n = X.shape[0]
    # Шаг 1: построение графа k-ближайших соседей
    tree = cKDTree(X)
    # distances и indices для k+1 ближайшего (включая саму точку)
    distances, indices = tree.query(X, k=n_neighbors+1)
    # Строим матрицу смежности в разреженном формате
    row_ind = []
    col_ind = []
    data = []
    for i in range(n):
        for j_idx, j in enumerate(indices[i]):
            if i == j:
                continue
            # делаем граф симметричным: добавляем ребро, если j сосед i или наоборот
            # проще добавить все ребра из k ближайших (кроме себя)
            row_ind.append(i)
            col_ind.append(j)
            data.append(distances[i][j_idx])
    adj = csr_matrix((data, (row_ind, col_ind)), shape=(n, n))
    # симметризуем (берём минимум, чтобы не было проблем с направленностью)
    adj = adj.minimum(adj.T)
    
    # Шаг 2: кратчайшие пути
    D_G = shortest_path(adj, directed=False)
    # Проверка связности
    n_components_graph, labels = connected_components(adj, directed=False)
    if n_components_graph > 1:
        print(f"Предупреждение: граф имеет {n_components_graph} компонент связности.")
        # Для несвязных компонент расстояния бесконечны; заменим на большое число
        inf_mask = np.isinf(D_G)
        max_finite = np.max(D_G[~inf_mask]) if np.any(~inf_mask) else 1.0
        D_G[inf_mask] = max_finite * 2  # условно большое расстояние
    
    # Шаг 3: двойное центрирование
    n = D_G.shape[0]
    J = np.eye(n) - np.ones((n, n)) / n
    B = -0.5 * J @ (D_G ** 2) @ J
    
    # Шаг 4: спектральное разложение
    eigvals, eigvecs = np.linalg.eigh(B)
    # Сортируем по убыванию
    idx = np.argsort(eigvals)[::-1]
    eigvals = eigvals[idx]
    eigvecs = eigvecs[:, idx]
    # Берём первые n_components положительных собственных значений
    # (могут быть небольшие отрицательные из-за неевклидовости)
    k = min(n_components, np.sum(eigvals > 1e-10))
    Y = eigvecs[:, :k] * np.sqrt(np.maximum(eigvals[:k], 0))
    return Y

# Генерация Swiss roll
n_samples = 1000
X_swiss, color = make_swiss_roll(n_samples=n_samples, noise=0.0)

# Применяем PCA и Isomap
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_swiss)
Y_isomap = isomap(X_swiss, n_neighbors=10, n_components=2)

# Визуализация
fig = plt.figure(figsize=(14, 4))
ax1 = fig.add_subplot(1, 3, 1, projection='3d')
ax1.scatter(X_swiss[:,0], X_swiss[:,1], X_swiss[:,2], c=color, cmap='Spectral', s=5)
ax1.set_title('Исходный Swiss roll (3D)')

ax2 = fig.add_subplot(1, 3, 2)
ax2.scatter(X_pca[:,0], X_pca[:,1], c=color, cmap='Spectral', s=5)
ax2.set_title('PCA')

ax3 = fig.add_subplot(1, 3, 3)
ax3.scatter(Y_isomap[:,0], Y_isomap[:,1], c=color, cmap='Spectral', s=5)
ax3.set_title('Isomap (k=10)')
plt.tight_layout()
plt.show()
```

На рисунке чётко видно: Isomap разворачивает рулет в плоский лист, сохраняя непрерывность цветового градиента. PCA же смешивает слои, и градиент рвётся.

### 4. Параметры и их влияние

Ключевой параметр Isomap — **число соседей $k$** (или радиус $\varepsilon$). Он определяет локальность аппроксимации и связность графа.

- **Слишком маленькое $k$** (например, $k=5$): граф может распасться на несколько компонент связности. Тогда геодезические расстояния между точками разных компонент не определены (бесконечны), и алгоритм не сможет построить единое глобальное вложение. На Swiss roll малые $k$ ведут к тому, что граф плохо перебрасывает мосты между витками спирали.
- **Слишком большое $k$** (например, $k=50$): граф становится почти полным, кратчайшие пути приближаются к прямым евклидовым расстояниям, и Isomap вырождается в обычный MDS на евклидовых расстояниях, теряя способность разворачивать многообразие. В пределе $k=n$ получаем PCA (если данные центрированы).
- **Оптимальное $k$** — достаточно большое, чтобы граф был связен и отражал геометрию многообразия, но не настолько, чтобы «срезать» изгибы. Обычно подбирается эмпирически.

Проверим влияние $k$ на Swiss roll:

```python
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
k_vals = [5, 10, 20, 50]
for ax, k in zip(axes, k_vals):
    Y = isomap(X_swiss, n_neighbors=k, n_components=2)
    ax.scatter(Y[:,0], Y[:,1], c=color, cmap='Spectral', s=5)
    ax.set_title(f'Isomap (k={k})')
plt.tight_layout()
plt.show()
```

При $k=5$ многие точки оказываются оторванными (видны отдельные облака). При $k=10$ рулет уже успешно разворачивается. При $k=50$ видно, что разворачивание становится менее чётким, появляется небольшая деформация.

Связность графа легко проверить: `scipy.sparse.csgraph.connected_components` возвращает число компонент. Если компонент больше одной, следует увеличить $k$ или использовать модификации (например, объединять ближайшие компоненты).

### 5. Сложность и масштабируемость

Вычислительная сложность Isomap складывается из трёх основных частей:

- **Поиск $k$ ближайших соседей:** с использованием KD‑дерева — $O(n d \log n)$ для каждой точки, в целом $O(n d \log n)$.
- **Кратчайшие пути:** алгоритм Дейкстры на $n$ вершин и примерно $nk$ рёбрах даёт $O(n \cdot (n k \log n)) = O(n^2 k \log n)$. На практике для $n \le 5000$ он приемлем.
- **Спектральное разложение матрицы $\mathbf{B}$:** $O(n^3)$, если вычислять все собственные значения, но для первых $k \ll n$ можно использовать итерационные методы (Lanczos), что снижает до $O(k n^2)$.

Основное узкое место — матрица $\mathbf{D}_G$ размера $n \times n$, хранящая все попарные геодезические расстояния. При $n=10\,000$ она уже занимает около 800 МБ (float64), что делает Isomap непрактичным для больших выборок. В таких случаях применяют приближения (Landmark Isomap) или другие методы (t‑SNE, UMAP), которые не требуют $O(n^2)$ памяти.

### 6. Ограничения и альтернативы

- **Выпуклость многообразия:** Isomap предполагает, что многообразие изометрично выпуклому подмножеству $\mathbb{R}^k$. Если многообразие имеет дыры или сложную топологию, геодезические расстояния могут искажаться, и вложение будет неадекватным.
- **Чувствительность к выбросам:** одна-единственная точка, соединённая длинным ребром с остальными, может создать «короткий путь» в графе, который грубо исказит геодезические расстояния. Выбросы следует удалять или использовать робастные варианты.
- **Шум:** даже при отсутствии выбросов малый шум может сделать граф менее похожим на истинное многообразие.
- **Альтернативы:** для нелинейной визуализации с акцентом на локальные структуры и кластеры популярны t‑SNE и UMAP. Локально линейное встраивание (LLE) также сохраняет локальные свойства, но не использует геодезические расстояния.

### 7. Сравнение на реальных данных: Digits

Применим Isomap к набору рукописных цифр `load_digits()` (8×8 пикселей, 64 признака). Сравним визуализацию с PCA.

```python
digits = load_digits()
X_digits = digits.data
y_digits = digits.target

# Стандартизируем данные (важно для kNN)
from sklearn.preprocessing import StandardScaler
X_scaled = StandardScaler().fit_transform(X_digits)

# PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# Isomap
Y_isomap = isomap(X_scaled, n_neighbors=15, n_components=2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12,5))
for digit in np.unique(y_digits):
    ax1.scatter(X_pca[y_digits==digit,0], X_pca[y_digits==digit,1], label=str(digit), s=10)
ax1.set_title('PCA')
ax1.legend()

for digit in np.unique(y_digits):
    ax2.scatter(Y_isomap[y_digits==digit,0], Y_isomap[y_digits==digit,1], label=str(digit), s=10)
ax2.set_title('Isomap (k=15)')
ax2.legend()
plt.tight_layout()
plt.show()
```

Isomap обычно даёт более разделённые кластеры цифр, особенно для таких классов, как «1» и «7», которые в PCA могут накладываться. Это демонстрирует способность Isomap улавливать нелинейные вариации (например, толщину штриха, наклон).

### 8. Резюме

Isomap — элегантное расширение MDS на случай нелинейных многообразий. Он успешно разворачивает такие структуры, как Swiss roll, если правильно выбран параметр $k$. Однако алгоритм ограничен размером выборки из-за хранения полной матрицы расстояний и вычисления кратчайших путей, что делает его непрактичным для $n > 10\,000$. Тем не менее, его идейная простота и связь с классическим шкалированием делают его важной вехой в истории методов снижения размерности и полезным инструментом для небольших наборов данных.

**Вопросы для самопроверки:**
1. Почему в Isomap для получения координат используется MDS, а не PCA? Что принципиально меняется при замене евклидовых расстояний на геодезические?
2. Предположим, что данные равномерно распределены на поверхности сферы. Корректно ли Isomap отразит эту структуру в 2D? Поясните, какие проблемы могут возникнуть.
3. Как изменится результат Isomap, если вместо симметризации графа через минимум использовать среднее арифметическое расстояний? Объясните физический смысл.

## t‑SNE: t‑distributed Stochastic Neighbor Embedding

t‑SNE — один из самых популярных алгоритмов визуализации многомерных данных. В отличие от PCA и Isomap, он не стремится сохранить глобальные расстояния или развернуть многообразие. Его цель — отобразить **локальные соседства**: точки, которые были близки в исходном пространстве, должны остаться близкими в проекции, даже ценой искажения глобальной геометрии. Эта философия делает t‑SNE незаменимым для визуального поиска кластеров и аномалий, но накладывает важные ограничения, о которых нужно знать.

### 1. История и мотивация

Корни t‑SNE лежат в алгоритме SNE (Stochastic Neighbor Embedding), предложенном Джеффри Хинтоном и Сэмом Ровейсом в 2002 году. SNE преобразует евклидовы расстояния в исходном пространстве в вероятности, отражающие «сходство» точек: чем ближе точки, тем выше вероятность того, что одна выберет другую в качестве «соседа». Затем алгоритм ищет низкоразмерное представление, в котором распределение таких вероятностей максимально похоже на исходное в смысле дивергенции Кульбака–Лейблера.

Однако у классического SNE была серьёзная проблема: в низком измерении точки «скучывались» (crowding problem), потому что объём пространства мал, и трудно разместить удалённые точки достаточно далеко друг от друга, не нарушая локальную структуру. В 2008 году Лоренс ван дер Маатен и Джеффри Хинтон усовершенствовали SNE, заменив в низкоразмерном пространстве нормальное распределение на $t$-распределение Стьюдента с одной степенью свободы. Благодаря более тяжёлым хвостам $t$-распределение позволяет разнести несоседние точки дальше, решая проблему скученности. Так родился **t‑SNE**, ставший стандартом визуализации в bioinformatics, NLP и других областях.

### 2. Вероятностная модель в исходном пространстве

Пусть $\mathbf{X} = \{\mathbf{x}_1,\dots,\mathbf{x}_n\} \subset \mathbb{R}^d$ — исходные данные. Для каждой точки $\mathbf{x}_i$ t‑SNE вычисляет условную вероятность $p_{j|i}$, с которой $\mathbf{x}_i$ выбрала бы $\mathbf{x}_j$ в качестве соседа, если бы соседи выбирались пропорционально гауссовской плотности с центром в $\mathbf{x}_i$:

$$
p_{j|i} = \frac{\exp\!\bigl(-\|\mathbf{x}_i - \mathbf{x}_j\|^2 / (2\sigma_i^2)\bigr)}{\sum_{k \neq i} \exp\!\bigl(-\|\mathbf{x}_i - \mathbf{x}_k\|^2 / (2\sigma_i^2)\bigr)}, \quad p_{i|i} = 0.
$$

Параметр $\sigma_i$ индивидуален для каждой точки и подбирается так, чтобы условное распределение имело заданную **перплексию** (см. раздел 5). Затем вероятности симметризуются:

$$
p_{ij} = \frac{p_{j|i} + p_{i|j}}{2n}.
$$

Такая симметризация гарантирует, что сумма всех $p_{ij}$ равна единице, а каждая точка вносит равный вклад в целевую функцию. Матрица $\mathbf{P} = (p_{ij})$ трактуется как вероятностная мера попарного сходства в исходном пространстве.

### 3. Распределение в низкоразмерном пространстве

Низкоразмерное представление $\mathbf{Y} = \{\mathbf{y}_1,\dots,\mathbf{y}_n\} \subset \mathbb{R}^k$ (обычно $k=2$ или $3$) должно моделировать те же вероятности, но с другим распределением. Вместо гауссовского t‑SNE использует $t$-распределение Стьюдента с одной степенью свободы (Коши), чьи хвосты спадают как $1/(1+\|\mathbf{y}\|^2)$:

$$
q_{ij} = \frac{\bigl(1 + \|\mathbf{y}_i - \mathbf{y}_j\|^2\bigr)^{-1}}{\sum_{k \neq l} \bigl(1 + \|\mathbf{y}_k - \mathbf{y}_l\|^2\bigr)^{-1}}, \quad q_{ii}=0.
$$

Выбор $t$-распределения — ключевой момент. В низком измерении (особенно на плоскости) расстояния между точками ограничены, и если использовать гауссовское распределение, чтобы правильно разместить далёкие точки, локальные соседства разрушатся. $t$-распределение имеет существенно более тяжёлые хвосты: оно медленнее убывает с расстоянием. Это позволяет точкам, которые далеки в исходном пространстве, находиться далеко и в проекции, не вызывая при этом огромного штрафа, как в случае с гауссианой. С математической точки зрения, дисперсия $t$-распределения с одной степенью свободы бесконечна, что и решает crowding problem.

### 4. Функция потерь: дивергенция Кульбака–Лейблера

t‑SNE находит вложение $\mathbf{Y}$, минимизируя дивергенцию Кульбака–Лейблера между распределением $\mathbf{P}$ в исходном пространстве и распределением $\mathbf{Q}$ в низкоразмерном:

$$
C = \mathrm{KL}(\mathbf{P} \parallel \mathbf{Q}) = \sum_{i \neq j} p_{ij} \log \frac{p_{ij}}{q_{ij}}.
$$

Эта функция асимметрична и по-разному штрафует ошибки:
- Если $p_{ij}$ велико (точки близки в исходном пространстве), а $q_{ij}$ мало (они далеко в проекции), то $p_{ij}\log(p_{ij}/q_{ij})$ велико — **сильный штраф**.
- Если $p_{ij}$ мало, а $q_{ij}$ велико (далёкие точки стали близкими в проекции), штраф невелик.

Таким образом, t‑SNE в первую очередь старается сохранить локальную структуру, жертвуя глобальной.

### 5. Перплексия и подбор $\sigma_i$

Перплексия — один из важнейших гиперпараметров t‑SNE. Для каждой точки $\mathbf{x}_i$ величина $\sigma_i$ подбирается бинарным поиском так, чтобы эффективное число соседей, измеренное через энтропию, равнялось заданному значению:

$$
\operatorname{Perp}(P_i) = 2^{H(P_i)}, \quad H(P_i) = -\sum_{j \neq i} p_{j|i} \log_2 p_{j|i}.
$$

Перплексия примерно соответствует числу ближайших соседей, которые учитываются при построении вероятностного распределения. Типичные значения лежат в диапазоне от 5 до 50. Малая перплексия заставляет алгоритм фокусироваться на очень локальной структуре — результат может выглядеть как множество разрозненных кластеров даже для непрерывных данных. Слишком большая перплексия учитывает более глобальные связи, что может «схлопнуть» кластеры в одно облако. Часто по умолчанию берут `perplexity=30`.

### 6. Градиентный спуск

Минимизация $\mathrm{KL}(\mathbf{P} \parallel \mathbf{Q})$ выполняется градиентным спуском. Градиент функции потерь по точке $\mathbf{y}_i$ имеет компактный вид:

$$
\frac{\partial C}{\partial \mathbf{y}_i} = 4 \sum_{j \neq i} (p_{ij} - q_{ij}) (\mathbf{y}_i - \mathbf{y}_j) \bigl(1 + \|\mathbf{y}_i - \mathbf{y}_j\|^2\bigr)^{-1}.
$$

Первая часть $(p_{ij} - q_{ij})$ сравнивает исходную и текущую вероятности: если $p_{ij} > q_{ij}$, точки притягиваются, иначе отталкиваются. Множитель $(1 + \|\mathbf{y}_i - \mathbf{y}_j\|^2)^{-1}$ обеспечивает, что влияние точки затухает с расстоянием, стабилизируя оптимизацию.

На практике используется продвинутая схема: сначала точки $\mathbf{Y}$ инициализируются случайно (или PCA), затем применяется градиентный спуск с импульсом и адаптивной скоростью обучения. Ранняя фаза включает «раннюю преувеличение» (early exaggeration), когда все $p_{ij}$ умножаются на фиксированный коэффициент (например, 4 или 12), чтобы ускорить формирование глобальной структуры кластеров.

### 7. Код с sklearn и пример на Fashion‑MNIST

Рассмотрим применение t‑SNE к изображениям одежды из Fashion‑MNIST. Для ускорения возьмём подвыборку в 6000 объектов и стандартизируем признаки. Затем построим вложения с разной перплексией и сравним их визуально.

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import StandardScaler

# Загрузка Fashion‑MNIST (6000 точек для примера)
X, y = fetch_openml('Fashion-MNIST', version=1, return_X_y=True, as_frame=False)
X = X[:6000]
y = y[:6000].astype(int)

# Стандартизация
X_scaled = StandardScaler().fit_transform(X)

# Визуализация для разных перплексий
perplexities = [5, 30, 100]
fig, axes = plt.subplots(1, 3, figsize=(18,6))
for ax, perp in zip(axes, perplexities):
    tsne = TSNE(n_components=2, perplexity=perp, random_state=42, n_iter=1000, verbose=0)
    Y = tsne.fit_transform(X_scaled)
    ax.scatter(Y[:,0], Y[:,1], c=y, cmap='tab10', s=1)
    ax.set_title(f'Perplexity = {perp}')
plt.tight_layout()
plt.show()
```

Результат демонстрирует характерную зависимость: при `perplexity=5` кластеры дробятся на мелкие подгруппы, при `perplexity=30` структура читается наилучшим образом, при `perplexity=100` кластеры начинают сливаться в единое облако.

### 8. Недостатки t‑SNE и практические рекомендации

Несмотря на свою популярность, t‑SNE обладает рядом принципиальных ограничений, которые важно понимать, чтобы не сделать ложных выводов.

- **Стохастичность.** Результат зависит от случайного начального приближения и порядка обновления точек. Разные запуски даже с фиксированным `random_state` могут давать разные карты. При интерпретации следует убедиться, что найденная структура воспроизводится при нескольких запусках.

- **Глобальная структура не сохраняется.** Расстояния между кластерами, их взаимное расположение и размеры не несут информации. Нельзя утверждать, что кластер A дальше от B, чем от C, на основании расстояний в t‑SNE-проекции.

- **Вычислительная сложность.** Прямая реализация t‑SNE требует $O(n^2)$ памяти и времени из-за хранения всех попарных вероятностей. Для больших выборок применяют приближение Барнса–Хата (Barnes‑Hut) — $O(n \log n)$, доступное в sklearn через `method='barnes_hut'` (для данных с размерностью > 3). Ещё более масштабируемый вариант — FIt‑SNE, реализующий быстрое преобразование Фурье. Тем не менее, при $n > 10^5$ t‑SNE становится непрактичным даже с ускорениями.

- **Чувствительность к предобработке.** Поскольку t‑SNE опирается на евклидовы расстояния, признаки должны быть приведены к сопоставимому масштабу. Стандартизация (z‑score) почти обязательна. Также рекомендуется предварительно снизить размерность с помощью PCA (например, до 50 измерений), чтобы уменьшить шум и ускорить вычисления.

- **Ложные кластеры на шуме.** t‑SNE склонен находить структуру даже там, где её нет. Если применить t‑SNE к данным, состоящим из чистого шума (например, многомерное нормальное распределение), он всё равно выдаст некие кластеры. Это продемонстрировано ниже: t‑SNE может вводить в заблуждение, поэтому всегда нужна проверка альтернативными методами.

```python
# Демонстрация ложных кластеров на равномерном шуме
np.random.seed(42)
X_noise = np.random.randn(1000, 50)
tsne_noise = TSNE(n_components=2, perplexity=30, random_state=42)
Y_noise = tsne_noise.fit_transform(X_noise)
plt.scatter(Y_noise[:,0], Y_noise[:,1], s=5)
plt.title('t‑SNE на случайном шуме (50 признаков)')
plt.show()
```

- **Неспособность к обобщению (transduction).** t‑SNE не строит явной функции отображения, её невозможно применить к новым точкам без переобучения всей модели. Для обработки новых данных существуют параметрические версии (parametric t‑SNE), но они выходят за рамки классического алгоритма.

### 9. Советы по применению

1. **Всегда стандартизируйте данные.** Признаки с большим размахом будут доминировать в вычислении расстояний.
2. **Предварительно используйте PCA** для снижения шума и ускорения. Типичная рекомендация — уменьшить размерность до 50 перед t‑SNE.
3. **Не интерпретируйте глобальную структуру.** Размеры и расстояния между кластерами в проекции не отражают плотность или сходство классов.
4. **Экспериментируйте с перплексией.** Запустите t‑SNE с несколькими значениями (5, 30, 100) и сравните. Стабильная структура — признак её реальности.
5. **Фиксируйте `random_state`** для воспроизводимости и при сравнении разных гиперпараметров.
6. **Проверяйте, не создаёт ли t‑SNE кластеры из шума**, сравнив с PCA или визуализацией случайных данных.

### 10. Упражнения

1. *«Загрузите датасет с 50 независимыми признаками (равномерный шум) и примените t‑SNE с перплексией 30. Увидите ли вы кластеры? Какие выводы можно сделать о надёжности визуализации?»*
2. *«Почему при интерпретации t‑SNE нельзя полагаться на размер кластеров и расстояния между ними? Приведите пример, где это приводит к неверным выводам.»*
3. *«Используя Fashion‑MNIST, сравните визуализации PCA (первые 2 компоненты) и t‑SNE. Какие классы лучше разделяются каждым методом? Объясните причину.»*

Таким образом, t‑SNE — мощный, но капризный инструмент. Понимание его математики и внутренних механизмов позволяет избежать ловушек и извлечь максимум пользы из визуализации сложных данных. В следующем разделе мы рассмотрим UMAP, который во многом преодолевает недостатки t‑SNE, сохраняя как локальную, так и глобальную структуру, и обладает лучшей масштабируемостью.

## UMAP: Uniform Manifold Approximation and Projection

После детального знакомства с t‑SNE возникает естественный вопрос: можно ли сохранить его способность выделять локальные кластеры, но при этом получить лучшую масштабируемость, способность к обобщению и более честное отражение глобальной структуры? Ответом стал алгоритм **UMAP**, предложенный Л. Макиннесом и соавторами в 2018 году. UMAP (Uniform Manifold Approximation and Projection) основан на строгом математическом фундаменте — римановой геометрии и алгебраической топологии, но его практическая реализация проста, быстра и интуитивно понятна. Сегодня UMAP часто выбирают для визуализации больших коллекций данных, таких как миллионы клеток в single‑cell RNA‑seq, благодаря его скорости и качеству.

### 1. Мотивация создания UMAP

К моменту появления UMAP в 2018 году t‑SNE уже более десяти лет был стандартом визуализации многомерных данных. Однако накопились серьёзные претензии:

- **Масштабируемость.** Даже с аппроксимацией Барнса‑Хата t‑SNE с трудом обрабатывал более 100 000 точек.
- **Память.** Хранение полных попарных вероятностей ($O(n^2)$) делало невозможным работу с действительно большими выборками.
- **Глобальная структура.** t‑SNE фокусируется исключительно на локальных соседствах; расстояния и взаимное расположение кластеров не несут информации.
- **Невозможность добавить новые точки.** Отсутствие явного отображения означает, что для каждого нового набора данных нужно заново запускать алгоритм.

Авторы UMAP поставили цель: сохранить сильные стороны t‑SNE, но построить модель с более прочной математической основой, которая бы эффективно сохраняла и локальную, и глобальную структуру, работала на порядки быстрее и позволяла бы трансформировать новые данные.

### 2. Ключевые идеи UMAP (интуитивное введение)

В основе UMAP лежит предположение, что данные лежат на некотором неизвестном многообразии, и что на этом многообразии существует равномерная мера (риманова метрика). Задача — построить низкоразмерное представление, которое приближает топологическую структуру многообразия, закодированную в **взвешенном графе соседства**.

**Построение графа в исходном пространстве.**  
Для каждой точки $\mathbf{x}_i$ алгоритм находит её $k$ ближайших соседей (обычно $k=15$). Затем вычисляются направленные веса рёбер:

$$
v_{j|i} = \exp\!\Bigl( -\frac{\max(0, \, d(\mathbf{x}_i,\mathbf{x}_j) - \rho_i)}{\sigma_i} \Bigr).
$$

Здесь $d(\cdot,\cdot)$ — выбранная метрика (по умолчанию евклидова), $\rho_i$ — расстояние от $\mathbf{x}_i$ до её **самого близкого соседа** (это обеспечивает локальную адаптивность), а $\sigma_i$ подбирается так, чтобы суммарный вес $v_{j|i}$ по всем соседям был равен $\log_2 k$ (аналог перплексии, но для фиксированной ширины ядра). После этого веса симметризуются:

$$
w_{ij} = v_{j|i} + v_{i|j} - v_{j|i} v_{i|j}.
$$

Получается неориентированный взвешенный граф, где веса $w_{ij} \in [0,1]$ интерпретируются как вероятность того, что соответствующее ребро существует в некотором топологическом представлении данных.

**Граф в низкоразмерном пространстве.**  
В целевом пространстве $\mathbb{R}^k$ (обычно $k=2$ или $3$) UMAP определяет близость точек с помощью гладкой функции:

$$
\phi(\mathbf{y}_i, \mathbf{y}_j) = \bigl(1 + a \|\mathbf{y}_i - \mathbf{y}_j\|^{2b}\bigr)^{-1},
$$

где параметры $a$ и $b$ подбираются по заданному параметру `min_dist` — минимальному расстоянию, на котором точки могут находиться друг от друга в проекции. По форме эта кривая напоминает $t$-распределение Стьюдента, используемое в t‑SNE, но с более гибкой настройкой.

**Функция потерь: кросс‑энтропия.**  
В отличие от t‑SNE, который минимизирует KL‑дивергенцию между распределениями $p_{ij}$ и $q_{ij}$, UMAP минимизирует бинарную кросс‑энтропию между весами графа в исходном пространстве ($w_{ij}$) и функциями сходства в низкоразмерном ($\phi_{ij}$):

$$
\mathcal{L} = \sum_{i,j} \left[ w_{ij} \log \frac{w_{ij}}{\phi_{ij}} + (1 - w_{ij}) \log \frac{1 - w_{ij}}{1 - \phi_{ij}} \right].
$$

Первое слагаемое действует как сила **притяжения** между точками, которые были соседями в исходном пространстве, а второе — как сила **отталкивания** для несоседних точек. Благодаря второму слагаемому UMAP активно расталкивает далёкие точки, что помогает сохранить глобальную структуру.

**Оптимизация с отрицательной выборкой.**  
Прямой расчёт всех $n^2$ пар невозможен для больших $n$. UMAP использует приём, заимствованный из word2vec: на каждом шаге градиентного спуска рассматриваются только существующие рёбра графа (положительные примеры) и небольшое количество случайно выбранных несоседних точек (отрицательные примеры). Это снижает сложность одной эпохи до $O(n k)$, где $k$ — число соседей, и делает алгоритм сублинейным на практике.

### 3. Алгоритмические отличия от t‑SNE

- **Разреженный граф вместо полной матрицы.** UMAP строит граф $k$ ближайших соседей, а не вычисляет все попарные вероятности. Это радикально снижает затраты памяти и времени.
- **Локальная метрика с $\rho_i$.** Расстояние до ближайшего соседа вычитается, что делает меру инвариантной к локальной плотности: в областях с высокой плотностью эффективное расстояние увеличивается, в разреженных — уменьшается. Это даёт более равномерное представление.
- **Кросс‑энтропия вместо KL‑дивергенции.** Симметричная кросс‑энтропия сильнее штрафует за сближение далёких точек, что сохраняет глобальную структуру.
- **Отрицательная выборка.** Вместо вычисления всех $q_{ij}$ на каждом шаге, UMAP обновляет лишь малую часть несоседних пар, что даёт огромный выигрыш в скорости.
- **Возможность добавить новые точки.** После построения вложения можно обучить отображение (фактически, найти координаты для новых точек), минимизируя ту же кросс‑энтропию относительно уже зафиксированных точек. В реализации `umap-learn` для этого используется аппроксимация с помощью метода опорных векторов или градиентного спуска.

### 4. Преимущества UMAP перед t‑SNE

Подведём итог:

- **Скорость.** На практике UMAP работает за $O(n^{1.14})$ благодаря разреженному графу и отрицательной выборке. На 100 000 точек он тратит минуты, тогда как t‑SNE — часы.
- **Память.** Хранится только граф соседства, а не полная матрица $n \times n$.
- **Сохранение глобальной структуры.** Кластеры в UMAP располагаются более осмысленно относительно друг друга, сохраняются континуумы.
- **Воспроизводимость и стабильность.** Хотя случайное начальное приближение всё ещё влияет на результат, UMAP меньше подвержен образованию ложных кластеров из шума.
- **Добавление новых точек.** Метод `transform` позволяет проецировать ранее не виденные данные в то же пространство, что важно для промышленных пайплайнов.

### 5. Код с библиотекой umap‑learn и сравнение с t‑SNE

Установим `umap-learn` (`pip install umap-learn`) и сравним оба алгоритма на наборе цифр `digits` (1797 изображений 8×8). Сравним время выполнения и визуализацию.

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
import umap
import time

# Загрузка и стандартизация
digits = load_digits()
X, y = digits.data, digits.target
X_scaled = StandardScaler().fit_transform(X)

# t‑SNE
start = time.time()
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
Y_tsne = tsne.fit_transform(X_scaled)
time_tsne = time.time() - start
print(f"t‑SNE: {time_tsne:.2f} с")

# UMAP
start = time.time()
reducer = umap.UMAP(n_neighbors=15, min_dist=0.3, random_state=42)
Y_umap = reducer.fit_transform(X_scaled)
time_umap = time.time() - start
print(f"UMAP: {time_umap:.2f} с")

# Визуализация
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12,6))
ax1.scatter(Y_tsne[:,0], Y_tsne[:,1], c=y, cmap='tab10', s=10)
ax1.set_title('t‑SNE')
ax2.scatter(Y_umap[:,0], Y_umap[:,1], c=y, cmap='tab10', s=10)
ax2.set_title('UMAP')
plt.tight_layout()
plt.show()
```

На относительно маленьком `digits` t‑SNE может быть даже быстрее из-за константных факторов, но при увеличении размера данных разрыв становится драматическим (см. ниже).

### 6. Гиперпараметры UMAP

Два главных гиперпараметра:

- **`n_neighbors`** (по умолчанию 15). Определяет, насколько локальной или глобальной будет структура. Маленькие значения (5–10) концентрируются на тонкой локальной топологии: могут разделить даже мелкие подгруппы внутри класса. Большие значения (100–200) заставляют алгоритм учитывать более крупномасштабные связи, что делает карту более глобальной и сглаженной.
- **`min_dist`** (по умолчанию 0.1). Минимальное расстояние между точками в низкоразмерном пространстве. Низкие значения (0.0) дают очень плотные кластеры, высокие (0.99) размазывают точки равномернее. Этот параметр влияет исключительно на эстетику визуализации, а не на реальную структуру.

Продемонстрируем влияние `min_dist` и `n_neighbors` на примере `digits`.

```python
fig, axes = plt.subplots(2, 2, figsize=(12,10))
for ax, n_neigh, min_d in zip(axes.flat, [5, 5, 100, 100], [0.1, 0.9, 0.1, 0.9]):
    um = umap.UMAP(n_neighbors=n_neigh, min_dist=min_d, random_state=42)
    Y = um.fit_transform(X_scaled)
    ax.scatter(Y[:,0], Y[:,1], c=y, cmap='tab10', s=5)
    ax.set_title(f'n_neighbors={n_neigh}, min_dist={min_d}')
plt.tight_layout()
plt.show()
```

Видно, что при `n_neighbors=5` кластеры более раздроблены, при `n_neighbors=100` — более слитны и показывают глобальные связи. Параметр `min_dist` управляет плотностью: при 0.9 точки равномерно распределены.

### 7. Сравнительная таблица

| Характеристика               | t‑SNE                         | UMAP                          |
|-----------------------------|-------------------------------|-------------------------------|
| Сложность                   | $O(n^2)$ / $O(n \log n)$ с BH    | $O(n^{1.14})$ на практике     |
| Память                      | высокая (полные попарные вероятности) | низкая (разреженный граф)    |
| Глобальная структура        | плохо сохраняется             | хорошо сохраняется            |
| Добавление новых точек      | невозможно                    | возможно (transform)          |
| Ложные кластеры на шуме    | появляются                    | значительно меньше            |
| Основные параметры          | perplexity, early exaggeration| n_neighbors, min_dist, metric |
| Математическая основа       | вероятностная                 | топологическая + риманова геометрия |

### 8. Недостатки UMAP

- **Больше гиперпараметров.** Хотя на практике `n_neighbors` и `min_dist` подбираются быстро, новичку требуется больше экспериментов, чем с единственной перплексией в t‑SNE.
- **Топологическая теория.** Понимание, что именно делает алгоритм, требует знакомства с симплициальными множествами и римановой геометрией. Это может затруднить интерпретацию для пользователей без математического бэкграунда.
- **Случайность.** Результат варьирует от запуска к запуску, хотя и меньше, чем у t‑SNE. Для воспроизводимости следует всегда фиксировать `random_state`.

### 9. Пример на большом датасете: Fashion‑MNIST

Загрузим 20 000 изображений из Fashion‑MNIST и сравним время работы t‑SNE (с Barnes‑Hut) и UMAP.

```python
from sklearn.datasets import fetch_openml

X_fashion, y_fashion = fetch_openml('Fashion-MNIST', version=1, return_X_y=True, as_frame=False)
X_fashion = X_fashion[:20000]
y_fashion = y_fashion[:20000].astype(int)
X_f_std = StandardScaler().fit_transform(X_fashion)

# t‑SNE
start = time.time()
Y_f_tsne = TSNE(n_components=2, method='barnes_hut', random_state=42).fit_transform(X_f_std)
print(f"t‑SNE (Barnes‑Hut) 20k: {time.time()-start:.2f} с")

# UMAP
start = time.time()
Y_f_umap = umap.UMAP(n_neighbors=15, min_dist=0.2, random_state=42).fit_transform(X_f_std)
print(f"UMAP 20k: {time.time()-start:.2f} с")
```

На таком объёме преимущество UMAP по скорости становится подавляющим: разница может достигать одного–двух порядков.

### 10. Резюме: когда использовать t‑SNE, а когда UMAP?

- **Выбирайте UMAP**, если:
  - У вас более 10 000 объектов,
  - важна глобальная структура (например, плавные переходы между классами),
  - нужно проецировать новые точки (transform),
  - требуется высокая скорость при приемлемом качестве.

- **Выбирайте t‑SNE**, если:
  - выборка мала (< 5000) и хочется максимально детальной локальной структуры,
  - вы работаете в среде, где t‑SNE уже хорошо отлажен,
  - интерпретация важнее, чем глобальная картина (с оговоркой про ложные кластеры).

Оба метода не являются заменой PCA — они служат для визуализации и исследования данных, а не для построения прогностических моделей. В реальном пайплайне их часто применяют **после** снижения размерности PCA до 50 компонент, что сглаживает шум и ускоряет сходимость.

### 11. Задания для самопроверки

1. *Загрузите датасет `fetch_openml('CIFAR-100')` (или его подмножество), примените PCA, t‑SNE и UMAP для визуализации 5000 изображений. Какие классы разделяются лучше каждым методом?*
2. *Используйте `umap.UMAP().fit_transform` и `umap.UMAP().fit().transform` на одном и том же наборе данных. Сравните полученные координаты для тренировочных и тестовых точек. Насколько хорошо UMAP обобщается?*
3. *Экспериментально определите, как меняется время выполнения UMAP при изменении `n_neighbors` от 5 до 200 на фиксированной выборке из 50 000 точек. Постройте график и прокомментируйте результат.*

Таким образом, UMAP является мощным современным инструментом, удачно сочетающим скорость, качество и математическую красоту. Его появление значительно расширило возможности анализа данных, и сегодня он входит в золотой стандарт инструментов data scientist'а. В следующем разделе мы кратко затронем метод Isomap, с которого началась история нелинейного снижения размерности.

## Практические рекомендации: выбор метода снижения размерности и типичные ошибки

Мы рассмотрели три класса методов — линейный PCA, геодезический Isomap, вероятностный t‑SNE и топологический UMAP. Теперь, имея в арсенале понимание их сильных и слабых сторон, пора перейти к главному вопросу практикующего аналитика: **какой метод выбрать в своей задаче и как не допустить типичных ошибок?** В этом заключительном разделе мы построим дерево решений, дадим сравнительную таблицу, разберём распространённые промахи с примерами кода и предложим универсальный шаблон для быстрой оценки качества вложения.

### 1. Дерево принятия решений

Следующий алгоритм поможет быстро сузить круг подходящих методов. Двигайтесь по вопросам сверху вниз.

1. **Нужна ли интерпретируемость компонент?**  
   Если важно понимать, какие исходные признаки формируют оси, и строить причинные выводы → **PCA** (возможно, с ортогональным вращением типа varimax).  
   Иначе → идём дальше.

2. **Надо ли быстро добавлять новые точки после построения вложения?**  
   Если да — **PCA** или **UMAP** (через `.transform()`). t‑SNE и Isomap этого не умеют без переобучения.

3. **Объём данных > 50 000 точек?**  
   Да → **UMAP** (или сильно ускоренный t‑SNE с FIt‑SNE/OpenTSNE, но UMAP проще).  
   Нет → продолжаем.

4. **Задача — в первую очередь визуализация для разведочного анализа (кластеры, аномалии)?**  
   Да → **t‑SNE** (малые выборки) или **UMAP** (средние и большие).  
   Нет → может быть достаточно PCA.

5. **Много шума, сомнения в линейной природе данных?**  
   Да → **UMAP** (или t‑SNE, но предварительно отфильтровать шум через PCA до 30–50 компонент).  
   Нет → можно пробовать Isomap.

6. **Хотите сохранить глобальные расстояния (например, для построения карты, где расстояние между группами отражает сходство)?**  
   Да → **UMAP**, **Isomap** или **PCA** (если структура линейна).  
   Нет → t‑SNE (глобальные расстояния не интерпретируются).

7. **Данные лежат на изогнутом многообразии без дырок, и выборка мала (n < 2000)?**  
   Да → **Isomap** может дать красивое разворачивание.

Это дерево покрывает большинство ситуаций. Если сомневаетесь — запустите PCA, t‑SNE и UMAP параллельно и сравните визуально и количественно (см. раздел 6).

### 2. Сравнительная таблица по ключевым критериям

| Метод  | Глобальная структура | Локальная структура | Скорость (n=10⁴) | Добавление новых точек | Интерпретируемость |
|--------|----------------------|---------------------|-------------------|------------------------|---------------------|
| PCA    | ★★★★★ (линейная)      | ★☆☆☆☆               | ★★★★★ (доля секунды) | да                     | высокая (нагрузки)  |
| Isomap | ★★★★☆ (геодезическая) | ★★★★☆               | ★☆☆☆☆ (куб, память) | нет                    | средняя (осей нет)  |
| t‑SNE  | ★☆☆☆☆ (не сохраняется) | ★★★★★               | ★★★☆☆ (Barnes‑Hut)  | нет                    | низкая (оси не интерпретируются) |
| UMAP   | ★★★★☆ (хорошо)        | ★★★★★               | ★★★★★ (сублинейно)  | да (transform)         | низкая              |

### 3. Типичные ошибки и как их избежать

#### 3.1 Забыли масштабирование признаков

Все методы, основанные на евклидовых расстояниях, чувствительны к разнице масштабов признаков. Если один признак измеряется в тысячах, а другой — в долях, то первый подавит все остальные. Вывод: всегда применяйте `StandardScaler` (или хотя бы `MinMaxScaler`) перед PCA, t‑SNE, UMAP, Isomap.

```python
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
```

**Демонстрация:** сгенерируем данные с двумя признаками, размах которых отличается в 100 раз. Покажем, что без масштабирования PCA полностью теряет второй признак, а с масштабированием — оба вносят вклад.

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

np.random.seed(0)
X1 = np.random.normal(0, 1, 200)
X2 = np.random.normal(0, 100, 200)
X_raw = np.column_stack([X1, X2])

# Без масштабирования
pca_raw = PCA().fit(X_raw)
# Со стандартизацией
X_scaled = StandardScaler().fit_transform(X_raw)
pca_scaled = PCA().fit(X_scaled)

fig, axes = plt.subplots(1,2, figsize=(10,4))
axes[0].bar([1,2], pca_raw.explained_variance_ratio_)
axes[0].set_title('PCA без масштабирования')
axes[1].bar([1,2], pca_scaled.explained_variance_ratio_)
axes[1].set_title('PCA со стандартизацией')
for ax in axes:
    ax.set_xlabel('Компонента')
    ax.set_ylabel('Доля дисперсии')
plt.tight_layout()
plt.show()
```

Без масштабирования первая компонента забирает почти всю дисперсию (признак с большим размахом), вторая игнорируется. После стандартизации вклад выравнивается.

#### 3.2 Не фиксируют `random_state`

t‑SNE и UMAP — стохастические алгоритмы. Результат может меняться от запуска к запуску. При сравнении гиперпараметров обязательно фиксируйте `random_state`. Воспроизводимость важна и для публикаций, и для отладки.

```python
tsne = TSNE(random_state=42)
umap = umap.UMAP(random_state=42)
```

#### 3.3 Интерпретируют расстояния между кластерами в t‑SNE

Это, пожалуй, самая распространённая ошибка. В t‑SNE глобальная геометрия не сохраняется: два кластера, которые выглядят далеко на графике, в исходном пространстве могут быть близки, и наоборот. Никогда не интерпретируйте расстояние между кластерами и их размеры в t‑SNE как меру сходства.

#### 3.4 Применяют t‑SNE к сырым данным высокой размерности

Если число признаков 1000 или больше, t‑SNE будет медленным и шумным. Сначала примените PCA для сокращения размерности до 30–50, затем t‑SNE к полученным главным компонентам. Это ускоряет вычисления и часто улучшает визуализацию.

```python
pca = PCA(n_components=50)
X_pca = pca.fit_transform(X_scaled)
tsne = TSNE()
Y = tsne.fit_transform(X_pca)
```

#### 3.5 Слишком маленькая или большая перплексия в t‑SNE

Перплексия должна быть в разумных пределах (обычно от 5 до 50). При перплексии 2 алгоритм видит лишь пару ближайших соседей и создаёт множество разрозненных групп. При перплексии 200 он смотрит слишком глобально, кластеры слипаются. Экспериментируйте с диапазоном 10–50, типичное начальное значение — 30.

#### 3.6 Не проверяют связность графа в Isomap

Если граф соседей распадается, Isomap не может построить единое вложение. Перед запуском проверяйте число компонент связности и при необходимости увеличивайте `n_neighbors` или отбрасывайте мелкие компоненты.

```python
from scipy.sparse.csgraph import connected_components
from sklearn.neighbors import NearestNeighbors

k = 15
nn = NearestNeighbors(n_neighbors=k).fit(X_scaled)
adj = nn.kneighbors_graph(mode='distance')
n_comp, labels = connected_components(adj, directed=False)
if n_comp > 1:
    print(f"Внимание: граф имеет {n_comp} компонент связности")
```

#### 3.7 Используют UMAP с параметрами по умолчанию для всех датасетов

UMAP имеет два ключевых параметра: `n_neighbors` (15 по умолчанию) и `min_dist` (0.1). Для разных задач их надо подбирать. Маленький `n_neighbors` — детальная локальная структура, большой — глобальная. Параметр `min_dist` управляет компактностью кластеров; для плотных данных можно уменьшить до 0.0, для разреженных — увеличить до 0.5.

### 4. Предобработка данных для нелинейных методов

- **Удалите константные и почти константные признаки.** Они не несут информации, но могут создавать вырожденные расстояния.
- **Примените PCA для шумоподавления.** 30–50 главных компонент сохраняют большую часть сигнала, отсеивая шум, и делают матрицы плотнее.
- **Логарифмируйте сильно асимметричные признаки** (например, денежные суммы, количество упоминаний). Это уменьшает влияние выбросов.
- **Стандартизируйте** (z‑score) почти всегда. Если данные разреженные (bag of words), используйте `TfidfTransformer` или оставьте как есть, но тогда используйте косинусное расстояние в UMAP (`metric='cosine'`).

### 5. Как оценить качество снижения размерности без меток?

Без правильных ответов оценить вложение можно количественно с помощью метрик, основанных на сохранении соседства.

- **Trustworthiness** (сохраняет ли проекция ближайших соседей): для каждой точки проверяется, не попали ли в её окрестность в проекции те точки, которые были далеки в исходном пространстве.
- **Continuity** (сохраняется ли непрерывность групп): не потерялись ли истинные соседи в проекции.
- `sklearn.manifold.trustworthiness(X, Y, n_neighbors=5)` возвращает число от 0 до 1, где 1 — идеальное сохранение.

Для PCA основная метрика — кумулятивная объяснённая дисперсия.

Визуальная оценка остаётся важным, но субъективным критерием. Всегда полезно посмотреть на проекцию глазами, но подтверждать выбранное количество компонент метрикой.

### 6. Код‑шпаргалка: сравнение PCA, t‑SNE, UMAP на одном датасете

Ниже приведён скрипт, который загружает `digits`, применяет три метода, вычисляет время и метрику trustworthiness, и строит сравнительные графики. Его можно использовать как отправную точку в своих проектах.

```python
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE, trustworthiness
import umap
import time
import pandas as pd
import matplotlib.pyplot as plt

# Загрузка и стандартизация
digits = load_digits()
X, y = digits.data, digits.target
X_scaled = StandardScaler().fit_transform(X)

results = []
methods = {'PCA': PCA(n_components=2),
           't‑SNE': TSNE(n_components=2, perplexity=30, random_state=42),
           'UMAP': umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.3, random_state=42)}

for name, model in methods.items():
    start = time.time()
    Y = model.fit_transform(X_scaled)
    elapsed = time.time() - start
    trust = trustworthiness(X_scaled, Y, n_neighbors=10)
    results.append({'Метод': name, 'Время (с)': elapsed, 'Trustworthiness': trust})

print(pd.DataFrame(results).round(3))

# Визуализация
fig, axes = plt.subplots(1, 3, figsize=(16,5))
for ax, (name, model) in zip(axes, methods.items()):
    Y = model.fit_transform(X_scaled)
    ax.scatter(Y[:,0], Y[:,1], c=y, cmap='tab10', s=5)
    ax.set_title(name)
plt.tight_layout()
plt.show()
```

### 7. Ложные кластеры t‑SNE на шуме (демонстрация)

```python
np.random.seed(42)
X_noise = np.random.randn(500, 50)
Y_noise = TSNE(random_state=42).fit_transform(X_noise)
plt.scatter(Y_noise[:,0], Y_noise[:,1])
plt.title('t‑SNE на чистом шуме (500 точек, 50 признаков)')
plt.show()
```

На графике отчётливо видны структуры, хотя данные абсолютно случайны. Это должно стать предупреждением: всегда проверяйте гипотезу о неслучайности кластеров альтернативными методами.

### 8. Заключение и общие рекомендации

- Не существует единственного «лучшего» метода снижения размерности. Выбор зависит от цели (визуализация, предобработка, понимание), объёма данных, требований к интерпретируемости и масштабируемости.
- **Начинайте с PCA.** Это быстрый, детерминированный метод, который даёт представление о линейной структуре данных и позволяет оценить долю объяснённой дисперсии.
- **Для визуализации и разведочного анализа** используйте UMAP (основной рабочий инструмент) или t‑SNE (для малых данных, когда важна тонкая локальная детализация).
- **Isomap** хорош для учебных целей и на маленьких выборках с явной многообразной структурой (Swiss roll), но не масштабируется.
- **Всегда масштабируйте признаки**, фиксируйте `random_state`, проверяйте устойчивость результатов.
- **Не интерпретируйте глобальные расстояния в t‑SNE** и размеры кластеров.
- Применяйте количественные метрики (`trustworthiness`) наряду с визуальной оценкой.
- Следуйте правилу: «Сначала PCA до 50 компонент, затем UMAP/t‑SNE». Это улучшит и ускорит результат.
- Помните о ложных кластерах и проверяйте гипотезы с помощью нескольких запусков и методов.

### 9. Упражнения для закрепления

1. **Задача на выбор метода:**  
   У вас есть данные о покупках клиентов (5000 записей, 200 признаков). Нужно визуализировать сегменты клиентов для презентации менеджменту. Какой метод выберете и почему?

2. **Эксперимент с faces:**  
   Загрузите датасет `fetch_olivetti_faces()` (400 лиц, 4096 пикселей). Примените PCA, t‑SNE и UMAP для визуализации. Какая визуализация лучше разделяет людей? Почему?

3. **Синтетический тест:**  
   Создайте датасет из двух вложенных спиралей (как в `make_swiss_roll`, но с двумя отдельными листами). Какой метод (Isomap, t‑SNE, UMAP) лучше разделит спирали? Проверьте с помощью `trustworthiness`.

4. **Поиск оптимальных параметров:**  
   Для набора данных `load_digits()` постройте тепловую карту trustworthiness в зависимости от `n_neighbors` (10, 15, 30, 50) и `min_dist` (0.0, 0.1, 0.3, 0.5) для UMAP. Сделайте вывод о наилучших параметрах.

Эта глава вооружила вас полным спектром инструментов снижения размерности — от классического PCA до современных UMAP и t‑SNE. Теперь вы способны осознанно выбирать метод, настраивать его и избегать ловушек, свойственных даже опытным аналитикам. В следующей главе мы перейдём к методам кластеризации, где умение правильно представить данные играет ключевую роль.